# SemBlend Autoresearch Analysis

Multi-metric visualization of autonomous SemBlend experiments from `results.tsv`.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

# Load results (tab-separated, multi-metric format)
df = pd.read_csv("results.tsv", sep="\t")
df["primary_value"] = pd.to_numeric(df["primary_value"], errors="coerce")
df["status"] = df["status"].str.strip().str.upper()

# Parse secondary_metrics column into separate columns
def parse_secondary(s):
    if pd.isna(s) or not isinstance(s, str):
        return {}
    pairs = s.split(",")
    result = {}
    for pair in pairs:
        if "=" in pair:
            k, v = pair.split("=", 1)
            try:
                result[k.strip()] = float(v.strip())
            except ValueError:
                result[k.strip()] = v.strip()
    return result

secondary = df["secondary_metrics"].apply(parse_secondary)
sec_df = pd.DataFrame(secondary.tolist())
df = pd.concat([df, sec_df], axis=1)

print(f"Total experiments: {len(df)}")
print(f"Columns: {list(df.columns)}")
df.head(10)

In [ ]:
# Experiment outcomes
counts = df["status"].value_counts()
print("Experiment outcomes:")
print(counts.to_string())

n_keep = counts.get("KEEP", 0)
n_discard = counts.get("DISCARD", 0)
n_decided = n_keep + n_discard
if n_decided > 0:
    print(f"\nKeep rate: {n_keep}/{n_decided} = {n_keep / n_decided:.1%}")

## TTFT Speedup Progression

In [ ]:
# Filter to TTFT experiments
ttft = df[df["benchmark"].str.contains("ttft", na=False, case=False)].copy()
ttft_kept = ttft[ttft["status"] == "KEEP"].copy()

if len(ttft_kept) > 0:
    fig, ax = plt.subplots(figsize=(14, 7))
    
    # Plot primary metric (e.g., ttft_speedup_8k) over time
    ax.plot(range(len(ttft_kept)), ttft_kept["primary_value"], 'o-', 
            color='#2ecc71', markersize=8, label='TTFT Speedup (kept)')
    
    # Also plot discarded experiments as faint dots
    ttft_disc = ttft[ttft["status"] == "DISCARD"].copy()
    if len(ttft_disc) > 0:
        ax.scatter(range(len(ttft_disc)), ttft_disc["primary_value"],
                   c='#cccccc', s=20, alpha=0.5, label='Discarded')
    
    ax.set_xlabel('Experiment #', fontsize=12)
    ax.set_ylabel('TTFT Speedup (higher is better)', fontsize=12)
    ax.set_title('TTFT Speedup Progression', fontsize=14)
    ax.legend()
    ax.grid(True, alpha=0.2)
    plt.tight_layout()
    plt.savefig('progress_ttft.png', dpi=150, bbox_inches='tight')
    plt.show()
else:
    print('No TTFT experiments found yet.')

## Quality Metrics Trajectory

In [ ]:
# Filter to quality experiments
quality = df[df["benchmark"].str.contains("quality", na=False, case=False)].copy()

if len(quality) > 0:
    fig, axes = plt.subplots(1, 2, figsize=(16, 6))
    
    # ROUGE-L
    if "rouge_l" in quality.columns:
        q_valid = quality.dropna(subset=["rouge_l"])
        axes[0].plot(range(len(q_valid)), q_valid["rouge_l"], 'o-', color='#3498db')
        axes[0].axhline(y=0.95, color='red', linestyle='--', alpha=0.5, label='Target: 0.95')
        axes[0].set_ylabel('ROUGE-L')
        axes[0].set_title('ROUGE-L (higher is better)')
        axes[0].legend()
        axes[0].grid(True, alpha=0.2)
    
    # PPL Ratio
    if "ppl_ratio" in quality.columns:
        q_valid = quality.dropna(subset=["ppl_ratio"])
        axes[1].plot(range(len(q_valid)), q_valid["ppl_ratio"], 'o-', color='#e74c3c')
        axes[1].axhline(y=1.0, color='green', linestyle='--', alpha=0.5, label='Baseline: 1.0')
        axes[1].set_ylabel('PPL Ratio (SemBlend/Cold)')
        axes[1].set_title('PPL Ratio (closer to 1.0 is better)')
        axes[1].legend()
        axes[1].grid(True, alpha=0.2)
    
    for ax in axes:
        ax.set_xlabel('Experiment #')
    
    plt.tight_layout()
    plt.savefig('progress_quality.png', dpi=150, bbox_inches='tight')
    plt.show()
else:
    print('No quality experiments found yet.')

## Hit Rate Trends

In [ ]:
# Hit rate across all experiments
if "hit_rate" in df.columns:
    hr = df.dropna(subset=["hit_rate"]).copy()
    if len(hr) > 0:
        fig, ax = plt.subplots(figsize=(14, 5))
        kept = hr[hr["status"] == "KEEP"]
        disc = hr[hr["status"] == "DISCARD"]
        
        if len(disc) > 0:
            ax.scatter(disc.index, disc["hit_rate"], c='#cccccc', s=20, alpha=0.5, label='Discarded')
        if len(kept) > 0:
            ax.scatter(kept.index, kept["hit_rate"], c='#2ecc71', s=50, 
                       edgecolors='black', linewidths=0.5, label='Kept')
        
        ax.set_xlabel('Experiment #')
        ax.set_ylabel('Hit Rate')
        ax.set_title('Donor Hit Rate Across Experiments')
        ax.legend()
        ax.grid(True, alpha=0.2)
        plt.tight_layout()
        plt.show()
else:
    print('No hit rate data found yet.')

## Gap Checklist (from ANALYSIS.md)

In [ ]:
# Check which gaps have been addressed
gaps = {
    "Multi-model (2+ models)": df["description"].str.contains("llama|mistral|multi.model", case=False, na=False).any(),
    "Real-world datasets (3+)": df["description"].str.contains("multinews|wikihow|samsun|cnn|dailymail", case=False, na=False).any(),
    "Ablation studies": df["benchmark"].str.contains("ablation", case=False, na=False).any(),
    "Quality (max_tokens=256)": df["benchmark"].str.contains("quality", case=False, na=False).any() if "rouge_l" in df.columns else False,
    "Memory measurement": df["benchmark"].str.contains("memory", case=False, na=False).any(),
    "Donor scaling": df["benchmark"].str.contains("scale", case=False, na=False).any(),
}

print("Gap Checklist:")
for gap, filled in gaps.items():
    status = "FILLED" if filled else "OPEN"
    marker = "[x]" if filled else "[ ]"
    print(f"  {marker} {gap}: {status}")

## Summary Statistics

In [ ]:
kept = df[df["status"] == "KEEP"].copy()

# Per-benchmark summary
benchmarks = df["benchmark"].dropna().unique()
print("Per-benchmark summary:\n")
for bench in benchmarks:
    subset = df[df["benchmark"] == bench]
    kept_subset = subset[subset["status"] == "KEEP"]
    print(f"  {bench}:")
    print(f"    Total runs: {len(subset)}")
    print(f"    Kept: {len(kept_subset)}")
    if len(kept_subset) > 0:
        best = kept_subset.loc[kept_subset["primary_value"].idxmax()]
        print(f"    Best {best['primary_metric']}: {best['primary_value']}")
        print(f"    Best experiment: {best['description']}")
    print()